In [1]:
import sys
from pathlib import Path

sys.path.append(f"{Path().absolute().parent}")

In [2]:
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, module="gpytorch")

In [3]:
from apps.mobility_robustness_optimization.mobility_robustness_optimization import *
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO

In [4]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [5]:
topology = pd.read_csv("data/sim_data/topology.csv")
ue_data = pd.read_csv('data/sim_data/UE_Data_20UE_100ticks.csv')

topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

## Simple MRO

In [6]:
mro = SimpleMRO(params, topology)

In [7]:
mro.update(ue_data)

No Bayesian Digital Twins available for update. Training from scratch.


[2025-04-30 20:11:57,690] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)
[2025-04-30 20:11:57,730] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019080)
[2025-04-30 20:11:57,765] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019104)
[2025-04-30 20:11:57,802] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019142)
[2025-04-30 20:11:57,839] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019231)
[2025-04-30 20:11:57,875] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019352)
[2025-04-30 20:11:57,912] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019533)
[2025-04-30 20:11:57,950] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019720)
[2025-04-30 20:11:57,986] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019932)
[2025-04-30 20:11:58,024] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020132)
[2025-04-30 20:11:58,061] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020358)
[2025-04-30 20:11:58,099] INFO:  Iter 12/100 - Loss: 0.568 (delta=-0.020523)
[2025-04-30 20:11:58,135] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020744)
[2025-04-30 20

In [8]:
hyst,ttt = mro.solve()

/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      3.1221391137   7      100.000000  
1      0.9474688756   70     99.450000   
2      3.5534207635   57     99.450000   
3      4.1825611430   12     99.950000   
4      1.5255928590   11     100.000000  
5      3.3025013790   74     99.450000   
6      3.2391935549   28     99.750000   
7      1.1290855949   65     99.450000   
8      3.8235523497   11     100.000000  
9      1.2567039969   84     99.350000   
10     0.2017001008   86     99.250000   
11     1.9777435366   85     99.300000   
12     2.0634466779   65     99.450000   
13     3.0155269794   52     99.500000   
14     0.9244971233   33     99.650000   
15     2.8258129522   19     99.750000   
16     3.2637129322   95     99.250000   
17     0.5187812840   56     99.450000   
18     2.7155799732   76     99.400000   
19     1.9950440244   65     99.450000   
20     1.6077091132   26     99.750000   
21     2.7741018729   71     99.45

## RL MRO

In [9]:
topology = pd.read_csv("data/sim_data/topology.csv")
ue_data = pd.read_csv('data/sim_data/UE_Data_20UE_100ticks.csv')

In [10]:
mro = ReinforcedMRO(params, topology)

In [11]:
mro.update(ue_data)

No Bayesian Digital Twins available for update. Training from scratch.


[2025-04-30 20:14:54,113] INFO:  Iter 1/100 - Loss: 0.759 (delta=inf)
[2025-04-30 20:14:54,148] INFO:  Iter 2/100 - Loss: 0.741 (delta=-0.018533)
[2025-04-30 20:14:54,179] INFO:  Iter 3/100 - Loss: 0.722 (delta=-0.018642)
[2025-04-30 20:14:54,208] INFO:  Iter 4/100 - Loss: 0.703 (delta=-0.018757)
[2025-04-30 20:14:54,238] INFO:  Iter 5/100 - Loss: 0.684 (delta=-0.018918)
[2025-04-30 20:14:54,268] INFO:  Iter 6/100 - Loss: 0.665 (delta=-0.019123)
[2025-04-30 20:14:54,298] INFO:  Iter 7/100 - Loss: 0.646 (delta=-0.019324)
[2025-04-30 20:14:54,328] INFO:  Iter 8/100 - Loss: 0.626 (delta=-0.019506)
[2025-04-30 20:14:54,357] INFO:  Iter 9/100 - Loss: 0.607 (delta=-0.019744)
[2025-04-30 20:14:54,387] INFO:  Iter 10/100 - Loss: 0.587 (delta=-0.019916)
[2025-04-30 20:14:54,417] INFO:  Iter 11/100 - Loss: 0.567 (delta=-0.020109)
[2025-04-30 20:14:54,446] INFO:  Iter 12/100 - Loss: 0.546 (delta=-0.020306)
[2025-04-30 20:14:54,475] INFO:  Iter 13/100 - Loss: 0.526 (delta=-0.020495)
[2025-04-30 20

In [12]:
hyst,ttt = mro.solve()

/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Using cpu device
Episode: 1, Timestep: 1, Hyst: 0.000000, TTT: 2, Reward: 89.100000, Done: False
Episode: 1, Timestep: 2, Hyst: 0.000000, TTT: 2, Reward: 89.100000, Done: False
Episode: 1, Timestep: 3, Hyst: 0.000000, TTT: 2, Reward: 89.100000, Done: False
Episode: 1, Timestep: 4, Hyst: 0.744255, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 5, Hyst: 0.000000, TTT: 2, Reward: 89.100000, Done: False
Episode: 1, Timestep: 6, Hyst: 0.000000, TTT: 2, Reward: 89.100000, Done: False
Episode: 1, Timestep: 7, Hyst: 0.000000, TTT: 2, Reward: 89.100000, Done: False
Episode: 1, Timestep: 8, Hyst: 1.816199, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 9, Hyst: 0.708112, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 10, Hyst: 0.000000, TTT: 2, Reward: 89.100000, Done: False
Episode: 1, Timestep: 11, Hyst: 1.706758, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 12, Hyst: 1.151884, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep